# Binary: all features (colour + texture), model comparison, feature selection and hyperparameter optimisation

* Features: `features/train.csv` and `features/val.csv` (notebook 06), all groups except shape.
* Every model is a **Pipeline**, so all of it is fitted inside each CV fold on train only (no leakage):
  `drop constant → drop correlated → scale → [feature selection] → model`
* **Stage 1, screening (sections 2–4):** every model × feature selector with a small grid. This picks the **best selector for each model** quickly.
* **Stage 2, optimisation (section 5):** for each model, a **randomized search** over a wide hyperparameter space, including the selector's and the correlation filter's parameters, with its best selector.
* All choices use the **5-fold CV on train** (`cv_bal_acc`). `val` is only reported, never used to choose.
* Results: `Binary/results_models.csv` (screening), `Binary/results_tuned.csv` (optimisation), `models/binary_best.joblib`.

In [ ]:
import os, sys, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in ("Binary", "Multiclass", "notebooks") else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
SEED = 42

import warnings
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectPercentile, SelectFromModel, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score, roc_curve,
                             confusion_matrix, ConfusionMatrixDisplay)
from feature_extraction import feature_groups, select_groups, CorrelationFilter
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None
warnings.filterwarnings("ignore", category=UserWarning)

## 0. Configuration

In [ ]:
FEAT_DIR = PROJECT_ROOT / "features"
GROUPS = "all"              # "all" or a list, e.g. ["hist_hsv", "moments_lab", "glcm", "lbp", "gabor"]
EXCLUDE = []                # groups to leave out, e.g. ["hog"] (position dependent)

# which models / selectors to run (comment out to save time)
MODELS_TO_RUN = ["LogReg", "SVM-RBF", "kNN", "RandomForest", "ExtraTrees", "XGBoost", "LightGBM"]
SELECTORS_TO_RUN = ["none", "anova", "rf", "l1"]

CORR_THRESHOLD = 0.95       # drop features with |r| above this with an earlier one
N_JOBS = -1                 # GridSearchCV runs the folds / parameter sets in parallel
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

## 1. Load the features

In [ ]:
train, val = pd.read_csv(FEAT_DIR / "train.csv"), pd.read_csv(FEAT_DIR / "val.csv")
groups = feature_groups(train.columns) if GROUPS == "all" else list(GROUPS)
groups = [g for g in groups if g not in EXCLUDE]
F_train, F_val = select_groups(train, groups), select_groups(val, groups)
X_train, X_val = F_train.values.astype(np.float64), F_val.values.astype(np.float64)
y_train, y_val = train["y"].values, val["y"].values
FEATURE_NAMES = np.array(F_train.columns)
LABELS = ["nevus", "others"]
print(f"{X_train.shape[1]} features from {len(groups)} groups:", groups)
print("train:", np.bincount(y_train), "| val:", np.bincount(y_val))

## 2. Selectors and models
The grids are kept small on purpose. Tree models get `n_jobs=1` because `GridSearchCV` is already parallel.

In [ ]:
SELECTORS = {
    "none":  ("passthrough", {}),
    "anova": (SelectPercentile(f_classif), {"select__percentile": [10, 30]}),
    "rf":    (SelectFromModel(RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=1)),
              {"select__threshold": ["median", "mean"]}),
    "l1":    (SelectFromModel(LogisticRegression(penalty="l1", solver="liblinear", max_iter=5000)),
              {"select__estimator__C": [0.05, 0.2]}),
}

MODELS = {
    "LogReg":       (LogisticRegression(max_iter=5000), {"model__C": [0.001, 0.01, 0.1, 1]}),
    "SVM-RBF":      (SVC(random_state=SEED), {"model__C": [1, 10], "model__gamma": ["scale", 0.001]}),
    "kNN":          (KNeighborsClassifier(), {"model__n_neighbors": [11, 21, 41], "model__weights": ["uniform", "distance"]}),
    "RandomForest": (RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=1),
                     {"model__max_features": ["sqrt", 0.1], "model__min_samples_leaf": [1, 3]}),
    "ExtraTrees":   (ExtraTreesClassifier(n_estimators=500, random_state=SEED, n_jobs=1),
                     {"model__max_features": ["sqrt", 0.1], "model__min_samples_leaf": [1, 3]}),
}
if XGBClassifier is not None:
    MODELS["XGBoost"] = (XGBClassifier(n_estimators=400, subsample=0.8, colsample_bytree=0.5,
                                       eval_metric="logloss", random_state=SEED, n_jobs=1),
                         {"model__max_depth": [3, 6], "model__learning_rate": [0.05, 0.1]})
if LGBMClassifier is not None:
    MODELS["LightGBM"] = (LGBMClassifier(n_estimators=400, subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
                                         random_state=SEED, n_jobs=1, verbose=-1),
                          {"model__num_leaves": [15, 31], "model__learning_rate": [0.05, 0.1]})

def make_pipeline_for(selector, model):
    return Pipeline([
        ("var", VarianceThreshold(0.0)),
        ("corr", CorrelationFilter(CORR_THRESHOLD)),
        ("scale", StandardScaler()),
        ("select", selector),
        ("model", model),
    ])

def scores(est, X):
    """Continuous score for the AUC (probability if available, else decision function)."""
    return est.predict_proba(X)[:, 1] if hasattr(est, "predict_proba") else est.decision_function(X)

def n_selected(pipe):
    """Number of features that reach the model (after constant / correlation filters and selection)."""
    sel = pipe.named_steps["select"]
    if hasattr(sel, "get_support"):
        return int(sel.get_support().sum())
    return int(pipe.named_steps["corr"].get_support().sum())

print("models:", [m for m in MODELS_TO_RUN if m in MODELS], "| selectors:", SELECTORS_TO_RUN)
missing = [m for m in MODELS_TO_RUN if m not in MODELS]
if missing:
    print("not installed (skipped):", missing)

## 3. Stage 1: screening (every model × selector, small grids)
Quick on purpose. Its job is to choose the **feature selector for each model**. The real tuning is in stage 2.

In [ ]:
results, fitted = [], {}
for m in [m for m in MODELS_TO_RUN if m in MODELS]:
    model, mgrid = MODELS[m]
    for s in SELECTORS_TO_RUN:
        selector, sgrid = SELECTORS[s]
        gs = GridSearchCV(make_pipeline_for(selector, model), {**sgrid, **mgrid}, cv=cv,
                          scoring="balanced_accuracy", n_jobs=N_JOBS, error_score=np.nan)
        t0 = time.time()
        gs.fit(X_train, y_train)
        best = gs.best_estimator_
        pred = best.predict(X_val)
        tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
        results.append({"model": m, "selector": s,
                        "cv_bal_acc": round(gs.best_score_, 4),
                        "cv_std": round(gs.cv_results_["std_test_score"][gs.best_index_], 4),
                        "n_features": n_selected(best),
                        "val_bal_acc": round(balanced_accuracy_score(y_val, pred), 4),
                        "val_acc": round(accuracy_score(y_val, pred), 4),
                        "val_sens": round(tp / (tp + fn), 4), "val_spec": round(tn / (tn + fp), 4),
                        "val_auc": round(roc_auc_score(y_val, scores(best, X_val)), 4),
                        "best_params": str(gs.best_params_), "time_s": round(time.time() - t0)})
        fitted[(m, s)] = best
        r = results[-1]
        print(f"{m:13s} {s:6s} cv {r['cv_bal_acc']:.3f}±{r['cv_std']:.3f} | val {r['val_bal_acc']:.3f} "
              f"| {r['n_features']} feats | {r['time_s']} s")

res = pd.DataFrame(results).sort_values("cv_bal_acc", ascending=False).reset_index(drop=True)
res

## 4. Comparison

In [ ]:
pivot = res.pivot(index="model", columns="selector", values="cv_bal_acc")
fig, ax = plt.subplots(figsize=(1.6 * pivot.shape[1] + 2, 0.5 * pivot.shape[0] + 1.5))
im = ax.imshow(pivot.values, cmap="Greens")
ax.set_xticks(range(pivot.shape[1]), pivot.columns); ax.set_yticks(range(pivot.shape[0]), pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=9)
ax.set_title("CV balanced accuracy (train, 5 folds)"); plt.tight_layout(); plt.show()

## 5. Stage 2: hyperparameter optimisation per model
For every model in `TUNE_MODELS`, a `RandomizedSearchCV` with `N_ITER` random combinations from the spaces below (continuous parameters on a log scale), using the selector that was best for that model in stage 1. The selector's parameters and the correlation threshold are searched as well.

`N_ITER` controls the time: each combination is 5 fits. 40–60 is a good start; raise it for the 2–3 best models.

In [ ]:
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV

TUNE_MODELS = "all"     # "all" (every model of stage 1), "top3" (3 best of stage 1), or a list, e.g. ["SVM-RBF", "LightGBM"]
N_ITER = 50             # random combinations per model
SELECTOR_FOR = {}       # force a selector, e.g. {"kNN": "anova"}; otherwise the best of stage 1

SPACE_CORR = {"corr__threshold": [0.90, 0.95, 0.99]}
SPACE_SELECTOR = {
    "none":  {},
    "anova": {"select__percentile": [5, 10, 20, 30, 50, 75]},
    "rf":    {"select__threshold": ["0.5*mean", "median", "mean", "1.5*mean"]},
    "l1":    {"select__estimator__C": loguniform(0.01, 1)},
}
SPACE_MODEL = {
    "LogReg": {
        "model__C": loguniform(1e-4, 10),
        "model__penalty": ["l1", "l2"],
        "model__solver": ["liblinear"],
        "model__class_weight": [None, "balanced"],
    },
    "SVM-RBF": {
        "model__C": loguniform(1e-2, 1e3),
        "model__gamma": loguniform(1e-5, 1e-1),
        "model__class_weight": [None, "balanced"],
    },
    "kNN": {
        "model__n_neighbors": randint(3, 80),
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2],                       # 1 = Manhattan, 2 = Euclidean
    },
    "RandomForest": {
        "model__n_estimators": [300, 500, 800],
        "model__max_features": ["sqrt", "log2", 0.05, 0.1, 0.2, 0.3],
        "model__max_depth": [None, 8, 16, 32],
        "model__min_samples_leaf": randint(1, 10),
        "model__min_samples_split": randint(2, 20),
        "model__class_weight": [None, "balanced", "balanced_subsample"],
    },
    "ExtraTrees": {
        "model__n_estimators": [300, 500, 800],
        "model__max_features": ["sqrt", "log2", 0.05, 0.1, 0.2, 0.3],
        "model__max_depth": [None, 8, 16, 32],
        "model__min_samples_leaf": randint(1, 10),
        "model__min_samples_split": randint(2, 20),
        "model__class_weight": [None, "balanced"],
    },
    "XGBoost": {
        "model__n_estimators": randint(200, 1000),
        "model__learning_rate": loguniform(0.01, 0.3),
        "model__max_depth": randint(2, 8),
        "model__min_child_weight": loguniform(1, 20),
        "model__subsample": uniform(0.6, 0.4),          # 0.6 - 1.0
        "model__colsample_bytree": uniform(0.2, 0.6),   # 0.2 - 0.8
        "model__gamma": uniform(0, 5),
        "model__reg_lambda": loguniform(1e-3, 10),
        "model__reg_alpha": loguniform(1e-4, 1),
    },
    "LightGBM": {
        "model__n_estimators": randint(200, 1000),
        "model__learning_rate": loguniform(0.01, 0.3),
        "model__num_leaves": randint(8, 64),
        "model__min_child_samples": randint(5, 50),
        "model__subsample": uniform(0.6, 0.4),
        "model__colsample_bytree": uniform(0.2, 0.6),
        "model__reg_lambda": loguniform(1e-3, 10),
        "model__reg_alpha": loguniform(1e-4, 1),
    },
}

# which models, and with which selector
best_screen = res.sort_values("cv_bal_acc", ascending=False).drop_duplicates("model").set_index("model")
if TUNE_MODELS == "all":
    to_tune = best_screen.index.tolist()
elif TUNE_MODELS == "top3":
    to_tune = best_screen.index[:3].tolist()
else:
    to_tune = [m for m in TUNE_MODELS if m in MODELS]
plan = {m: SELECTOR_FOR.get(m, best_screen.loc[m, "selector"]) for m in to_tune}
print("models to tune -> selector:", plan)
print(f"fits: about {len(plan) * N_ITER * cv.get_n_splits()} (+ refits)")

In [ ]:
tuned, tuned_rows = {}, []
for m, s in plan.items():
    model, _ = MODELS[m]
    selector, _ = SELECTORS[s]
    space = {**SPACE_CORR, **SPACE_SELECTOR[s], **SPACE_MODEL[m]}
    rs = RandomizedSearchCV(make_pipeline_for(selector, model), space, n_iter=N_ITER, cv=cv,
                            scoring="balanced_accuracy", n_jobs=N_JOBS, random_state=SEED,
                            error_score=np.nan, return_train_score=True)
    t0 = time.time()
    rs.fit(X_train, y_train)
    best_m = rs.best_estimator_
    pred = best_m.predict(X_val)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    i_best = rs.best_index_
    tuned[m] = rs
    tuned_rows.append({
        "model": m, "selector": s,
        "cv_bal_acc": round(rs.best_score_, 4),
        "cv_std": round(rs.cv_results_["std_test_score"][i_best], 4),
        "train_bal_acc": round(rs.cv_results_["mean_train_score"][i_best], 4),   # >> cv = overfitting
        "screening_cv": best_screen.loc[m, "cv_bal_acc"] if m in best_screen.index else np.nan,
        "n_features": n_selected(best_m),
        "val_bal_acc": round(balanced_accuracy_score(y_val, pred), 4),
        "val_acc": round(accuracy_score(y_val, pred), 4),
        "val_sens": round(tp / (tp + fn), 4), "val_spec": round(tn / (tn + fp), 4),
        "val_auc": round(roc_auc_score(y_val, scores(best_m, X_val)), 4),
        "best_params": str({k: (round(v, 5) if isinstance(v, float) else v) for k, v in rs.best_params_.items()}),
        "time_s": round(time.time() - t0)})
    r = tuned_rows[-1]
    print(f"{m:13s} ({s:5s}) cv {r['screening_cv']:.3f} -> {r['cv_bal_acc']:.3f}±{r['cv_std']:.3f} "
          f"| train {r['train_bal_acc']:.3f} | val {r['val_bal_acc']:.3f} | {r['time_s']} s")

tuned_res = pd.DataFrame(tuned_rows).sort_values("cv_bal_acc", ascending=False).reset_index(drop=True)
tuned_res

### 5b. How each hyperparameter affects the CV score
For each tuned model, the CV score of every tried combination against each numeric hyperparameter. A clear trend, or a best value at the edge of the range, means the range should be moved or widened and the search repeated.

In [ ]:
for m, rs in tuned.items():
    cvr = pd.DataFrame(rs.cv_results_)
    params = [c for c in cvr.columns if c.startswith("param_")]
    numeric = [c for c in params if pd.to_numeric(cvr[c], errors="coerce").notna().mean() > 0.8]
    if not numeric:
        continue
    fig, axes = plt.subplots(1, len(numeric), figsize=(3.2 * len(numeric), 2.8), squeeze=False)
    for ax, c in zip(axes[0], numeric):
        x = pd.to_numeric(cvr[c], errors="coerce")
        ax.scatter(x, cvr["mean_test_score"], s=14, alpha=0.7)
        ax.scatter(x[rs.best_index_], cvr["mean_test_score"][rs.best_index_], s=60, c="r", marker="*")
        if x.min() > 0 and x.max() / x.min() > 50:
            ax.set_xscale("log")
        ax.set_title(c.replace("param_", "").replace("model__", ""), fontsize=8)
    axes[0][0].set_ylabel("CV bal. acc"); fig.suptitle(m, fontsize=10)
    plt.tight_layout(); plt.show()

## 6. Best model (by CV after tuning)

In [ ]:
# best = highest CV score after tuning (chosen WITHOUT looking at val)
bm, bs = tuned_res.loc[0, "model"], tuned_res.loc[0, "selector"]
best = tuned[bm].best_estimator_
print(f"best by CV: {bm} + {bs}  cv {tuned_res.loc[0, 'cv_bal_acc']}  ->  val bal. acc "
      f"{tuned_res.loc[0, 'val_bal_acc']}, AUC {tuned_res.loc[0, 'val_auc']}")
print("params:", tuned_res.loc[0, "best_params"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
ConfusionMatrixDisplay.from_predictions(y_val, best.predict(X_val), display_labels=LABELS,
                                        cmap="Blues", colorbar=False, ax=axes[0])
axes[0].set_title(f"{bm} + {bs} (val)")
for m, rs in tuned.items():
    sc = scores(rs.best_estimator_, X_val)
    fpr, tpr, _ = roc_curve(y_val, sc)
    axes[1].plot(fpr, tpr, label=f"{m} AUC {roc_auc_score(y_val, sc):.3f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=0.8); axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].legend(fontsize=7); axes[1].set_title("ROC on val (tuned models)")
plt.tight_layout(); plt.show()

## 7. Which feature groups matter?
**Group permutation importance** of the best pipeline on val: all the columns of one group are shuffled together, and the drop in balanced accuracy is measured. This is for **interpretation only**; it is not used to choose anything. It also shows which selected features survived.

In [ ]:
rng = np.random.default_rng(SEED)
base = balanced_accuracy_score(y_val, best.predict(X_val))
imp = []
for g in groups:
    cols = [i for i, c in enumerate(FEATURE_NAMES) if c.split("__", 1)[0] == g]
    drops = []
    for _ in range(5):
        Xp = X_val.copy()
        Xp[:, cols] = Xp[rng.permutation(len(Xp))][:, cols]
        drops.append(base - balanced_accuracy_score(y_val, best.predict(Xp)))
    imp.append({"group": g, "n_features": len(cols), "importance": np.mean(drops), "std": np.std(drops)})
imp = pd.DataFrame(imp).sort_values("importance", ascending=False)
ax = imp.plot.barh(x="group", y="importance", xerr="std", legend=False, figsize=(6, 0.3 * len(imp) + 1))
ax.invert_yaxis(); ax.set_xlabel("drop in val balanced accuracy when the group is shuffled")
plt.tight_layout(); plt.show()

# features kept by the selector, per group
keep = best.named_steps["var"].get_support()
names = FEATURE_NAMES[keep][best.named_steps["corr"].get_support()]
sel = best.named_steps["select"]
if hasattr(sel, "get_support"):
    names = names[sel.get_support()]
print("features used by the best pipeline per group:")
print(pd.Series([n.split("__", 1)[0] for n in names]).value_counts().to_string())

## 8. Save results and the best model

In [ ]:
import joblib
tag = "all" if GROUPS == "all" else "+".join(groups)
exc = "+".join(EXCLUDE)

def append(df, path):
    df = df.assign(groups=tag, excluded=exc, n_input_features=X_train.shape[1])
    if path.exists():
        old = pd.read_csv(path)
        old = old[~((old["groups"] == tag) & (old["excluded"].fillna("") == exc))]
        df = pd.concat([old, df], ignore_index=True)
    df.to_csv(path, index=False)

append(res, PROJECT_ROOT / "Binary" / "results_models.csv")       # stage 1
append(tuned_res, PROJECT_ROOT / "Binary" / "results_tuned.csv")  # stage 2

(PROJECT_ROOT / "models").mkdir(exist_ok=True)
joblib.dump({"pipeline": best, "groups": groups, "feature_names": FEATURE_NAMES.tolist(),
             "model": bm, "selector": bs, "params": tuned[bm].best_params_},
            PROJECT_ROOT / "models" / "binary_best.joblib")
print("saved: results_models.csv, results_tuned.csv, models/binary_best.joblib")